In [1]:
import pandas as pd

df = pd.read_csv("../datasets/fertilizer_data.csv")
df.head()

,Temparature,Humidity,Moisture,Soil Type,Crop Type,Nitrogen,Potassium,Phosphorous,Fertilizer Name
0,26,52,38,Sandy,Maize,37,0,0,Urea
1,29,52,45,Loamy,Sugarcane,12,0,36,DAP
2,34,65,62,Black,Cotton,7,9,30,14-35-14
3,32,62,34,Red,Tobacco,22,0,20,28-28
4,28,54,46,Clayey,Paddy,35,0,0,Urea


In [4]:
df.shape

(99, 9)

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 99 entries, 0 to 98
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   Temparature      99 non-null     int64
 1   Humidity         99 non-null     int64
 2   Moisture         99 non-null     int64
 3   Soil Type        99 non-null     str  
 4   Crop Type        99 non-null     str  
 5   Nitrogen         99 non-null     int64
 6   Potassium        99 non-null     int64
 7   Phosphorous      99 non-null     int64
 8   Fertilizer Name  99 non-null     str  
dtypes: int64(6), str(3)
memory usage: 7.1 KB


In [6]:
df.describe()

,Temparature,Humidity,Moisture,Nitrogen,Potassium,Phosphorous
count,99.000000,99.000000,99.000000,99.000000,99.000000,99.000000
mean,30.282828,59.151515,43.181818,18.909091,3.383838,18.606061
std,3.502304,5.840331,11.271568,11.599693,5.814667,13.476978
min,25.000000,50.000000,25.000000,4.000000,0.000000,0.000000
25%,28.000000,54.000000,34.000000,10.000000,0.000000,9.000000
50%,30.000000,60.000000,41.000000,13.000000,0.000000,19.000000
75%,33.000000,64.000000,50.500000,24.000000,7.500000,30.000000
max,38.000000,72.000000,65.000000,42.000000,19.000000,42.000000


In [7]:
df['Fertilizer Name'].value_counts()

Fertilizer Name
Urea        22
DAP         18
28-28       17
14-35-14    14
20-20       14
17-17-17     7
10-26-26     7
Name: count, dtype: int64

In [8]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['Fertilizer_Label'] = le.fit_transform(df['Fertilizer Name'])

print(df[['Fertilizer Name', 'Fertilizer_Label']].drop_duplicates().sort_values('Fertilizer_Label'))

   Fertilizer Name  Fertilizer_Label
63        10-26-26                 0
2         14-35-14                 1
5         17-17-17                 2
6            20-20                 3
3            28-28                 4
1              DAP                 5
0             Urea                 6


In [9]:
df_encoded = pd.get_dummies(df, columns=['Soil Type', 'Crop Type'])
df_encoded = df_encoded.drop(columns=['Fertilizer Name'])

X = df_encoded.drop(columns=['Fertilizer_Label'])
y = df_encoded['Fertilizer_Label']

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(X_train.shape, X_test.shape)

(79, 22) (20, 22)


In [10]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print()
print(classification_report(y_test, y_pred, target_names=le.classes_))

Accuracy: 1.0

              precision    recall  f1-score   support

    10-26-26       1.00      1.00      1.00         1
    14-35-14       1.00      1.00      1.00         3
    17-17-17       1.00      1.00      1.00         1
       20-20       1.00      1.00      1.00         3
       28-28       1.00      1.00      1.00         3
         DAP       1.00      1.00      1.00         4
        Urea       1.00      1.00      1.00         5

    accuracy                           1.00        20
   macro avg       1.00      1.00      1.00        20
weighted avg       1.00      1.00      1.00        20



In [11]:
import pandas as pd

importances = pd.Series(model.feature_importances_, index=X.columns)
importances.sort_values(ascending=False).head(15)

Phosphorous            0.307066
Nitrogen               0.228015
Potassium              0.140302
Moisture               0.069222
Temparature            0.056142
Humidity               0.052384
Crop Type_Sugarcane    0.014169
Soil Type_Black        0.013325
Soil Type_Red          0.013300
Soil Type_Loamy        0.013217
Crop Type_Millets      0.011960
Soil Type_Clayey       0.011333
Soil Type_Sandy        0.008753
Crop Type_Tobacco      0.008167
Crop Type_Pulses       0.007975
dtype: float64

In [12]:
df.groupby('Fertilizer Name')[['Nitrogen', 'Phosphorous', 'Potassium']].mean().round(1)

,Nitrogen,Phosphorous,Potassium
Fertilizer Name,,,
10-26-26,7.6,17.7,17.7
14-35-14,8.2,29.6,8.6
17-17-17,12.1,13.1,13.0
20-20,11.2,11.6,0.0
28-28,22.6,21.1,0.0
DAP,12.9,38.4,0.0
Urea,38.4,0.0,0.0


In [13]:
import joblib

final_model = RandomForestClassifier(n_estimators=100, random_state=42)
final_model.fit(X, y)
joblib.dump(final_model, "../models/fertilizer_model.pkl")
joblib.dump(le, "../models/fertilizer_label_encoder.pkl")

print("Both saved.")

Both saved.
